# Notebook 2 · Question 3: baseline and models
*Updated version of the team's first Question 3 notebook (P Khosa, `Baseline&Models.ipynb`; the original is kept in `notebooks/archive/`).*
The structure is the same (baseline, four models, comparison), with three corrections so it matches the rest of the project:
1. it uses the **time-based split** of Question 2.3 (train 2018–2021, validation 2022) instead of a random 70/15/15 split;
2. the **test year is not used** here (it is used once, in Question 5.3);
3. only the Question 2.4 features are inputs (`financial_year_end.year` and the raw opinion columns are excluded).

In [1]:
import sys, json, logging, warnings
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import Image, Markdown, display

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
warnings.filterwarnings("ignore")
logging.disable(logging.WARNING)
pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 180)
pd.set_option("display.float_format", "{:.3f}".format)
RESULTS, FIGS = ROOT / "results", ROOT / "reports" / "figures"
show = lambda name, width=800: display(Image(filename=str(FIGS / name), width=width))

In [2]:
from sklearn.base import clone
from pipeline_q24 import load_split, NUMERIC_FEATURES, CATEGORICAL_FEATURES
from models import build_models, BASELINES, SEED
from train_q3 import score

X_train, y_train = load_split("train")
X_val, y_val = load_split("val")
print("Inputs:", NUMERIC_FEATURES + CATEGORICAL_FEATURES)
print(f"Train {X_train.shape} | validation {X_val.shape} | seed {SEED}")

Inputs: ['total_revenue', 'total_expenditure', 'operating_surplus', 'cash_coverage_ratio', 'collection_rate', 'debtor_days', 'repairs_maintenance_ratio', 'irregular_expenditure_ratio', 'grant_dependency', 'prior_audit_outcome']
Train (950, 10) | validation (244, 10) | seed 42


## 3.1 Baselines
* **Dummy (most frequent):** predicts *not_at_risk* for everyone; sets the floor.
* **Repeat last year's outcome:** predicts that year *t+1* repeats year *t*; shows how much skill comes from audit persistence alone.
Neither counts among the four substantive models.
## 3.2–3.3 Four models on identical data
Two traditional models (Logistic Regression, Random Forest) and two neural networks (an MLP and a residual tabular network), all inside the
same preprocessing pipeline and trained on the same rows with seed 42.

In [3]:
models = build_models()
fitted, rows = {}, []
for name, model in models.items():
    m = clone(model).fit(X_train, y_train)
    fitted[name] = m
    va = score(y_val, m.predict(X_val), m.predict_proba(X_val)[:, 1])
    rows.append({"model": name, "family": "baseline" if name in BASELINES else
                 ("neural" if name in ("MLP", "Residual Tabular NN") else "traditional"), **va})
results = pd.DataFrame(rows).set_index("model")
results

,family,accuracy,macro_f1,weighted_f1,precision_at_risk,recall_at_risk,roc_auc,pr_auc
model,,,,,,,,
Baseline (Dummy),baseline,0.578,0.366,0.423,0.000,0.000,0.500,0.422
Baseline (Repeat last year),baseline,0.889,0.887,0.889,0.865,0.874,0.887,0.809
Logistic Regression,traditional,0.889,0.887,0.889,0.865,0.874,0.928,0.922
Random Forest,traditional,0.881,0.878,0.881,0.856,0.864,0.913,0.894
MLP,neural,0.881,0.879,0.881,0.843,0.883,0.929,0.917
Residual Tabular NN,neural,0.881,0.879,0.881,0.843,0.883,0.927,0.920


In [4]:
# How often does each model agree with the "repeat last year" rule on the validation year?
rule = (X_val["prior_audit_outcome"] == "at_risk").astype(int).to_numpy()
pd.Series({n: (fitted[n].predict(X_val) == rule).mean() for n in models if n not in BASELINES}, name="agreement with rule")

Logistic Regression   1.000
Random Forest         0.975
MLP                   0.984
Residual Tabular NN   0.984
Name: agreement with rule, dtype: float64

## 3.4 Architecture, hyper-parameters and training details

In [5]:
for name, model in models.items():
    clf = model.named_steps["model"] if hasattr(model, "named_steps") else model
    params = {k: v for k, v in clf.get_params().items() if v is not None and v != "deprecated"}
    print(f"{name}: {type(clf).__name__} {params}")
for name in ["MLP", "Residual Tabular NN"]:
    clf = fitted[name].named_steps["model"]
    print(f"{name}: best epoch {clf.best_epoch_}, stopped at epoch {clf.n_epochs_run_} (early stopping, patience {clf.patience})")

Baseline (Dummy): DummyClassifier {'strategy': 'most_frequent'}
Baseline (Repeat last year): PriorOutcomeRule {}
Logistic Regression: LogisticRegression {'C': 1.0, 'class_weight': 'balanced', 'dual': False, 'fit_intercept': True, 'intercept_scaling': 1, 'l1_ratio': 0.0, 'max_iter': 2000, 'random_state': 42, 'solver': 'lbfgs', 'tol': 0.0001, 'verbose': 0, 'warm_start': False}
Random Forest: RandomForestClassifier {'bootstrap': True, 'ccp_alpha': 0.0, 'class_weight': 'balanced', 'criterion': 'gini', 'max_features': 'sqrt', 'min_impurity_decrease': 0.0, 'min_samples_leaf': 1, 'min_samples_split': 2, 'min_weight_fraction_leaf': 0.0, 'n_estimators': 200, 'n_jobs': -1, 'oob_score': False, 'random_state': 42, 'verbose': 0, 'warm_start': False}
MLP: TorchTabularClassifier {'arch': 'mlp', 'batch_size': 32, 'class_weight': 'balanced', 'dropout': 0.2, 'es_fraction': 0.15, 'hidden': (64, 32), 'lr': 0.001, 'max_epochs': 100, 'n_blocks': 1, 'patience': 10, 'random_state': 42, 'use_skip': True, 'weig

## 3.5 Rationale
Logistic Regression suits fewer than 1,000 rows and nine ratios: stable and explainable. Random Forest captures non-linear effects and is
robust to heavy-tailed values but can memorise. The MLP tests whether a flexible model extracts more from the ratios; the residual network
tests whether extra depth with a skip connection helps. **Result:** all four beat the dummy baseline clearly, but the "repeat last year" rule
already reaches macro-F1 0.887, and Logistic Regression is the provisional champion (Question 4.3).